# 02 — Canonical Telecom data model

This notebook applies the synthetic-PON source adapter and then the shared
canonical contract. Native names are replaced by semantic metric identifiers;
topology is effective-dated; invalid and clipped observations remain explicit.

The model-facing run contains `SPEC-CORE` and `SPLITS` only. Evaluation truth
is not mounted here.


## 1. Setup


In [ ]:
from pathlib import Path
import os
import sys


if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")


def find_repository(start=Path.cwd()):
    """Find the checked-out repository when Jupyter starts in any subfolder."""
    override = os.getenv("TELCO_PROJECT_ROOT")
    if override:
        candidates = [Path(override).expanduser().resolve()]
    else:
        start = start.resolve()
        candidates = [start, *start.parents]
        if "google.colab" in sys.modules:
            candidates += [
                Path("/content/drive/MyDrive/anomaly_detection"),
                Path("/content/drive/MyDrive/telco-anomaly-detection"),
            ]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate.resolve()
    raise FileNotFoundError(
        "Open this notebook from the cloned repository, or set TELCO_PROJECT_ROOT."
    )


PROJECT_ROOT = find_repository()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from telco_anomaly.adapters.synthetic_pon import build_synthetic_pon_pack
from telco_anomaly.contract import (
    CORE_SCHEMAS, CORE_VERSION, OPTIONAL_CORE_SCHEMAS,
    build_canonical, check_core, read_pack,
)
from telco_anomaly.io import load_config, read_json, resolve_data_root, resolve_dataset_source

DATA_ROOT = resolve_data_root()
SOURCE = resolve_dataset_source(
    "synthetic_pon", data_root=DATA_ROOT, project_root=PROJECT_ROOT
)
METRIC_CONFIG = load_config("metric_registry", project_root=PROJECT_ROOT)
TOPOLOGY_CONFIG = load_config("topology", project_root=PROJECT_ROOT)

PACK_RUN_ID = os.getenv("PON_PACK_RUN_ID", "synthetic_pon_pack_v2")
CORE_RUN_ID = os.getenv("PON_CORE_RUN_ID", "synthetic_pon_core_v2")
PACK_ROOT = DATA_ROOT / "prepared" / "synthetic_pon" / PACK_RUN_ID
RUN_ROOT = DATA_ROOT / "core" / "synthetic_pon" / CORE_RUN_ID
BATCH_ROWS = int(os.getenv("PON_BATCH_ROWS", "200000"))

display(pd.Series({
    "source": str(SOURCE),
    "pack_staging": str(PACK_ROOT),
    "model_facing_run": str(RUN_ROOT),
    "contract_version": CORE_VERSION,
}, name="value").to_frame())


## 2. Inspect the canonical schemas


In [ ]:
schemas = {**CORE_SCHEMAS, **OPTIONAL_CORE_SCHEMAS}
display(pd.DataFrame([
    {"table": name, "columns": ", ".join(columns)}
    for name, columns in schemas.items()
]))


## 3. Build or validate the PON pack

The source adapter owns native field names and PON topology semantics. The
pack is immutable: change the run ID when inputs or mappings change rather
than overwriting a previous result.


In [ ]:
if PACK_ROOT.exists():
    pack_manifest = read_pack(PACK_ROOT)
    print("Validated existing pack")
else:
    pack_manifest = build_synthetic_pon_pack(
        SOURCE,
        PACK_ROOT,
        metric_registry=METRIC_CONFIG,
        topology_config=TOPOLOGY_CONFIG,
        include_evaluation=True,
        batch_rows=BATCH_ROWS,
    )
    print("Built new pack")

display(pd.Series({
    "pack_version": pack_manifest["pack_version"],
    "sector": pack_manifest["sector"],
    "telemetry_rows": pack_manifest["core_row_counts"]["telemetry"],
    "metrics": len(pack_manifest["metric_ids"]),
    "topology": pack_manifest["capabilities"]["topology"],
    "operational_events": pack_manifest["capabilities"]["operational_events"],
    "evaluation_tables_staged": pack_manifest["evaluation_tables"],
}, name="value").to_frame())


## 4. Materialise model-safe canonical data

Canonicalisation is the same for every future Telecom pack. It validates keys,
derives observed entity/episode bounds and collection gaps, and writes compact
manifests. The audit scans one Parquet part at a time; it does not perform a
global in-memory sort. `include_evaluation=False` is intentional.


In [ ]:
if RUN_ROOT.exists():
    run_manifest = read_json(RUN_ROOT / "run_manifest.json")
    print("Using existing immutable canonical run")
else:
    run_manifest = build_canonical(
        PACK_ROOT, RUN_ROOT, include_evaluation=False
    )
    print("Built new canonical run")

audit = check_core(RUN_ROOT / "SPEC-CORE")
assert not (RUN_ROOT / "SPEC-EVAL").exists()
display(pd.Series(audit, name="value").to_frame())


## 5. Inspect all small canonical tables and a telemetry sample


In [ ]:
CORE_ROOT = RUN_ROOT / "SPEC-CORE"
core_manifest = read_json(CORE_ROOT / "manifest.json")

for name in [
    "metric_catalogue", "entity_registry", "observation_episodes",
    "collection_gaps", "topology_memberships", "operational_events",
]:
    path = CORE_ROOT / f"{name}.parquet"
    if path.exists():
        frame = pd.read_parquet(path)
        print(f"\n{name}: {len(frame):,} rows")
        display(frame.head(10))

telemetry_parts = sorted((CORE_ROOT / "telemetry").glob("part-*.parquet"))
print(f"\ntelemetry: {len(telemetry_parts):,} Parquet parts")
telemetry_sample = next(
    pq.ParquetFile(telemetry_parts[0]).iter_batches(batch_size=100)
).to_pandas()
display(telemetry_sample.head(12))


## 6. Canonical acceptance checks


In [ ]:
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")

assert catalogue["metric_id"].str.contains("\\.", regex=True).all()
assert set(telemetry_sample["quality_code"]) <= {"measured", "invalid", "clipped"}
assert set(telemetry_sample["metric_id"]) <= set(catalogue["metric_id"])
assert core_manifest["capabilities"]["topology"] is True
assert not (RUN_ROOT / "SPEC-EVAL").exists()

print("PASS — semantic metric IDs are canonical")
print("PASS — topology is model-visible and effective-dated")
if core_manifest["capabilities"]["operational_events"]:
    print("PASS — source-observable operational events were kept separate from telemetry")
else:
    print("PASS — no operational event stream was invented")
print("PASS — evaluation truth is absent from the model-facing run")
print("Next: 03_SPLITS_AND_TRUTH_LOCK.ipynb")
